In [ ]:
import requests
from bs4 import BeautifulSoup
import time
import csv

def scrape_kompas_ke_csv(keywords, page_start, page_end):
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}
    semua_data = []

    print("\nDATA SCRAPING")

    for kw in keywords:
        tag = kw.lower().replace(" ", "-")
        for page in range(page_start, page_end + 1):
            url = f"https://www.kompas.com/tag/{tag}" if page == 1 else f"https://www.kompas.com/tag/{tag}?page={page}"

            try:
                res = requests.get(url, headers=headers)
                soup = BeautifulSoup(res.text, 'html.parser')
                box = soup.find('div', class_='sectionBox') or soup

                links = {}
                for a in box.find_all('a', href=True):
                    raw_text = a.text.strip().split('\n')[0].strip()
                    if "kompas.com/read/" in a['href'] and len(raw_text) > 15:
                        links[a['href']] = raw_text

                jumlah_berita = len(links)

                if jumlah_berita > 0:
                    print(f"Keyword '{kw}' ada {jumlah_berita} artikel diambil di halaman {page}")
                else:
                    print(f"Keyword '{kw}' kosong (0 artikel) di halaman {page}")
                    break

                for link, judul in links.items():
                    isi_lengkap = scrape_isi_berita_multi_page(link, headers)
                    semua_data.append([kw, page, judul, link, isi_lengkap])
                    time.sleep(0.1)

            except Exception as e:
                print(f"Keyword '{kw}' di halaman {page} -> Error: {e}")
    if semua_data:
        with open('mentah_kompas.csv', 'w', encoding='utf-8', newline='') as f:
            writer = csv.writer(f)
            writer.writerow(['Keyword', 'Halaman Tag', 'Judul Berita', 'Link Berita', 'Isi Berita'])
            writer.writerows(semua_data)
        print("\nScraping selesai. Data disimpan di 'mentah_kompas.csv'")
    else:
        print("\nTidak ada data yang berhasil diambil.")

def scrape_isi_berita_multi_page(url, headers):
    if "?page=all" not in url:
        url = url + "?page=all"
    isi_total = []
    try:
        res = requests.get(url, headers=headers)
        if res.status_code == 200:
            soup = BeautifulSoup(res.text, 'html.parser')
            content_div = soup.find('div', class_='read__content')
            paragraphs = content_div.find_all('p') if content_div else soup.find_all('p')
            
            for p in paragraphs:
                text = p.text.strip()
                if text and not text.startswith("Baca juga:") and not text.startswith("Simak juga:"):
                    isi_total.append(text)
    except:
        pass
        
    return " ".join(isi_total)
if __name__ == "__main__":
    DAFTAR_KEYWORD = [
    'anak dianiaya', 'dianiaya', 'dianiaya istri', 'dianiaya orangtua', 'dianiaya pacar', 'dianiaya suami', 'dicekik', 'diculik', 
    'digebuki', 'dihajar', 'dijambret', 'dikeroyok', 'dlempari', 'disabet', 'disekap', 'disiksa', 'disiram air keras',  'disundut',
    'ditampar', 'ditendang', 'ditusuk', 'istri aniaya suami', 'kdrt',  'kekerasan anak', 'kekerasan dalam rumah tangga', 'kekerasan fisik', 
    'kekerasan jalanan', 'kekerasan pelajar', 'kekerasan perempuan', 'korban penganiayaan', 'membacok', 'menganiaya', 'pelaku penganiayaan',
    'pemukulan', 'pembacokan', 'penculikan', 'penganiayaan', 'penganiayaan anak', 'penganiayaan asisten', 'penganiayaan balita', 
    'penganiayaan berat', 'penganiayaan istri', 'penganiayaan suami', 'pengeroyokan', 'pengrusakan', 'penusukan', 'penyiksaan', 'penyekapan',
    'suami aniaya istri', 'tersangka penganiayaan', 'dirundung', 'dibully', 'pembullyan', 'perundungan','dipukul', 'dihantam', 'dijambak', 'diseret', 
    'diancam sajam','clurit', 'celurit', 'senjata tajam', 'sajam', 'carok', 'tawuran pelajar', 'klitih', 'menabrak sengaja','tawuran','visum dokter',
    'luka lebam', 'luka tusuk', 'luka bacok',]   
    scrape_kompas_ke_csv(DAFTAR_KEYWORD, page_start=1, page_end=50)

In [9]:
import glob
import pandas as pd
import re

kumpulan_ambigu = ['ambigu', 'bandit', 'ingon', 'plunder', 'laboratorium', 'bnn', 'bnnk', 'bnnp','tes urine', 'jalur tikus', 'lintas provinsi', 
    'lintas negara','korupsi','aliran dana ilegal', 'aset disita', 'kiriman ekspedisi', 'urine positif', 'prostitusi', 'porno','visum', 'autopsi', 
    'olah tkp', 'tim inafis','kedokteran forensik', 'kerangka', 'visum dokter', 'otopsi', 'bnn', 'bnnk', 'bnnp', 'satresnarkoba', 'laboratorium',
    'tengkorak', 'membusuk', 'meraba', 'meremas','polisi gadungan', 'tni gadungan','pesta seks','pesta gay','jaksa gadungan', 'praktik dokter ilegal',
    'balap liar dibubarkan', 'geng motor', 'tindakan asusila','pelecehank di Kereta','klinik kecantikan palsu', 'suntik silikon ilegal','bandit',
    'intimidasi', 'jaringan', 'menabrak sengaja','sekarat', 'mati lemas', 'jeratan di leher', 'bercak darah','ujaran kebencian', 'clurit', 'celurit',
    'senjata tajam', 'sajam','hate speech', 'hoaks','konten ilegal', 'bengkel bodong','kartel', 'timbun', 'cyber bullying', 'perundungan siber',
    'balap liar dibubarkan', 'mafia karantina', 'aset disita', 'dugaan korupsi','dihantam', 'dijambak', 'diseret', 'diancam sajam',
    'pemerintah korup', 'oknum korup', 'pamer harta korupsi','hoaks','senjata api ilegal', 'peluru tajam', 'baku tembak',
    'geng motor', 'pekerja migran ilegal', 'pmi ilegal','jual beli ginjal']

semua_file = glob.glob(r"D:\Download\doc\SK_WIDIASIH\dataset_kompas\mentah_kompas*.csv")

list_df = []
for f in semua_file:
    df_individual = pd.read_csv(f, sep=',')
    list_df.append(df_individual)

df_gabungan = pd.concat(list_df, ignore_index=True)
print(f"Total awal setelah gabung: {len(df_gabungan)} baris.")

df_gabungan.columns = df_gabungan.columns.str.strip() 

kolom_link = 'Link Berita'
if kolom_link in df_gabungan.columns:
    df_gabungan.drop_duplicates(subset=[kolom_link], keep='first', inplace=True)
else:
    df_gabungan.drop_duplicates(inplace=True)
print(f"Total setelah drop duplicate berita: {len(df_gabungan)} baris.")

pattern = '|'.join([r'\b' + re.escape(keyword) + r'\b' for keyword in kumpulan_ambigu])

df_bersih = df_gabungan[~df_gabungan['Keyword'].astype(str).str.contains(pattern, case=False, na=False, regex=True)]

df_bersih.to_csv(r"D:\Download\doc\SK_WIDIASIH\dataset_kompas\dataset_kompas_gabungan_bersih.csv", index=False)
print(f"Selesai! Total baris akhir data: {len(df_bersih)} baris.")

Total awal setelah gabung: 186312 baris.
Total setelah drop duplicate berita: 121345 baris.
Selesai! Total baris akhir data: 103187 baris.
